# 🧪 Week 1 Hands-On Lab — Introduction to Amazon Bedrock

**Cloud AI Practitioner Program — CAIP 01**  
**Instructor:** Justin Ossai, GenAI Specialist Solutions Architect

---

## Learning Objectives

By the end of this lab, you will:

1. **Explore** this Jupyter Notebook and understand its structure (markdown cells vs. code cells)
2. **Install** the required dependencies to interact with Amazon Bedrock
3. **Initialize** the Bedrock client and verify connectivity
4. **Invoke** the Bedrock API with a prompt and display the results

---

## Prerequisites

- An AWS account with access to Amazon Bedrock
- IAM permissions: `bedrock:InvokeModel` on the model you plan to use
- Model access enabled in the Bedrock console (us-east-1 recommended)
- Python 3.9+

> **Tip:** If you're running this in SageMaker Studio, boto3 is already installed. If you're running locally, the install cell below will handle it.

---
## Step 1 — Install Required Dependencies

We need `boto3` (the AWS SDK for Python) to communicate with Amazon Bedrock.  
Run the cell below to install/upgrade it.

In [ ]:
# Install/upgrade boto3 — the AWS SDK for Python
%pip install --upgrade boto3 --quiet

print("✅ boto3 installed successfully.")

In [ ]:
# Import the libraries we'll use throughout this lab
import boto3
import json

print(f"boto3 version: {boto3.__version__}")
print("✅ All imports successful.")

---
## Step 2 — Initialize the Bedrock Runtime Client

Amazon Bedrock exposes foundation models through the **Bedrock Runtime** API.  
We create a client pointing to `bedrock-runtime` in our chosen region.

> **Note:** If you're running in SageMaker Studio, the notebook's execution role provides credentials automatically. If you're running locally, make sure your AWS CLI is configured (`aws configure`) or set the `AWS_ACCESS_KEY_ID` and `AWS_SECRET_ACCESS_KEY` environment variables.

In [ ]:
# Set the AWS region — us-east-1 has the broadest model availability
AWS_REGION = "us-east-1"

# Create the Bedrock Runtime client
bedrock_runtime = boto3.client(
    service_name="bedrock-runtime",
    region_name=AWS_REGION,
)

print(f"✅ Bedrock Runtime client initialized in {AWS_REGION}.")

### Verify Connectivity

Let's also create a **Bedrock management client** to list the foundation models available in your account.  
This confirms that your credentials and permissions are working correctly.

In [ ]:
# Create a Bedrock management client to verify connectivity
bedrock_client = boto3.client(
    service_name="bedrock",
    region_name=AWS_REGION,
)

# List a few available foundation models to confirm access
response = bedrock_client.list_foundation_models()
models = response["modelSummaries"]

print(f"✅ Connection verified — {len(models)} foundation models available.\n")
print("Sample models:")
for m in models[:5]:
    print(f"  • {m['modelId']}  ({m['providerName']})")

---
## Step 3 — Invoke the Bedrock API

Now for the exciting part — let's send a prompt to a foundation model and get a response.

We'll use **Amazon Titan Text Express** (`amazon.titan-text-express-v1`), which is available by default in most accounts.

> **How it works:**  
> 1. We build a JSON payload with our prompt and generation parameters  
> 2. We call `invoke_model()` with the model ID and payload  
> 3. We parse the JSON response to extract the generated text

In [ ]:
print(bedrock_runtime.meta.region_name)

In [ ]:
bedrock = boto3.client("bedrock", region_name="us-east-1")
print(bedrock.list_foundation_models(byProvider="Amazon")["modelSummaries"][0])


In [ ]:
MODEL_ID = "amazon.nova-micro-v1:0"

prompt = "Explain what machine learning is in 3 simple sentences that a beginner would understand."

body = json.dumps({
    "messages": [
        {
            "role": "user",
            "content": [
                {"text": prompt}
            ]
        }
    ],
    "inferenceConfig": {
        "maxTokens": 256,
        "temperature": 0.7,
        "topP": 0.9
    }
})

response = bedrock_runtime.invoke_model(
    modelId=MODEL_ID,
    body=body,
    contentType="application/json",
    accept="application/json"
)

result = json.loads(response["body"].read())
generated_text = result["output"]["message"]["content"][0]["text"]

print("📝 Prompt:")
print(f"   {prompt}\n")
print("🤖 Bedrock Response:")
print(f"   {generated_text.strip()}")




In [ ]:
models = bedrock.list_foundation_models(byProvider="Amazon")["modelSummaries"]
for m in models:
    if "nova" in m["modelId"].lower():
        print(m["modelId"], m["modelLifecycle"]["status"])


---
## Step 4 — Experiment with Different Prompts

Let's wrap the invoke logic into a reusable function so you can easily try different prompts and parameters.

In [ ]:
def ask_bedrock(prompt, max_tokens=256, temperature=0.7, top_p=0.9):
    """Send a prompt to Amazon Nova Micro and return the response."""
    payload = {
        "messages": [
            {
                "role": "user",
                "content": [{"text": prompt}],
            }
        ],
        "inferenceConfig": {
            "max_new_tokens": max_tokens,
            "temperature": temperature,
            "top_p": top_p,
        },
    }

    response = bedrock_runtime.invoke_model(
        modelId=MODEL_ID,
        contentType="application/json",
        accept="application/json",
        body=json.dumps(payload),
    )

    result = json.loads(response["body"].read())
    return result["output"]["message"]["content"][0]["text"].strip()


print("✅ ask_bedrock() function ready. Try it below!")


In [ ]:
# Try it out — ask about AWS AI services
answer = ask_bedrock("What is Amazon SageMaker and why is it useful for machine learning?")
print("🤖 Response:\n")
print(answer)

In [ ]:
# Experiment with temperature — lower = more focused, higher = more creative
print("── Temperature 0.1 (focused) ──")
print(ask_bedrock("Give me one fun fact about AI.", temperature=0.1))

print("\n── Temperature 0.9 (creative) ──")
print(ask_bedrock("Give me one fun fact about AI.", temperature=0.9))

In [ ]:
# 🎯 YOUR TURN — Replace the prompt below with your own question!
my_prompt = "What is the difference between supervised and unsupervised learning?"

my_answer = ask_bedrock(my_prompt)
print(f"📝 Your Prompt: {my_prompt}\n")
print(f"🤖 Response:\n{my_answer}")

---
## 🎉 Lab Complete!

You've successfully:

1. ✅ Explored this Jupyter Notebook (markdown + code cells)
2. ✅ Installed dependencies and imported boto3
3. ✅ Initialized the Bedrock Runtime client and verified connectivity
4. ✅ Invoked the Bedrock API and displayed generated text

### What You Learned

- **Amazon Bedrock** provides serverless access to foundation models from multiple providers
- The `bedrock-runtime` client is used to invoke models via `invoke_model()`
- You send a JSON payload with your prompt and generation config (temperature, max tokens, topP)
- The response contains generated text that you parse from JSON
- **Temperature** controls creativity: lower = more deterministic, higher = more varied

### Next Steps

- Try different model IDs (e.g., `anthropic.claude-3-haiku-20240307-v1:0`) — note that each model has its own payload format
- Explore the [Bedrock User Guide](https://docs.aws.amazon.com/bedrock/latest/userguide/) for more capabilities
- Next week we'll work with Amazon Polly, Transcribe, and run SageMaker notebooks

---
*Cloud AI Practitioner Program — CAIP 01 | Week 1 Lab*